# 03 — Qwen SFT with LoRA / QLoRA

Fine-tunes Qwen2.5 (7B or 72B) on the RF domain SFT dataset using LoRA.

**Why LoRA over full-weight fine-tuning (like `gpu_finetune.py`)?**
- Domain injection only needs to update ~1% of parameters
- QLoRA (4-bit quantized) runs a 7B model on a single 24 GB GPU
- Full-weight training for a 72B model needs 8× A100s — LoRA does it on 2×
- Qwen's general reasoning is preserved (LoRA doesn't overwrite base weights)

**Model choices:**
- `Qwen/Qwen2.5-7B-Instruct` — fast iteration, fits on 1× RTX 3090/4090
- `Qwen/Qwen2.5-72B-Instruct` — best quality, needs 2–4× A100 80GB with QLoRA
- `Qwen/Qwen2.5-Coder-7B-Instruct` — better for GLayout code generation tasks

In [ ]:
!pip install transformers peft trl bitsandbytes accelerate datasets wandb

In [ ]:
import os, json
from pathlib import Path
import torch

PROC_DIR = Path('../data/processed')

# --- CONFIG ---
MODEL_ID = 'Qwen/Qwen2.5-7B-Instruct'   # change to 72B for production
OUTPUT_DIR = Path('../data/models/qwen_rf_lora')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

USE_QLORA = True      # 4-bit quantization — set False if you have ≥80GB VRAM
LORA_R = 64           # LoRA rank: higher = more capacity, more memory
LORA_ALPHA = 128      # LoRA scaling factor (usually 2×r)
LORA_DROPOUT = 0.05
MAX_SEQ_LEN = 4096
BATCH_SIZE = 2        # per GPU
GRAD_ACCUM = 8        # effective batch = BATCH_SIZE × GRAD_ACCUM × n_gpus
LR = 2e-4
EPOCHS = 3
WARMUP_RATIO = 0.05

print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB' if torch.cuda.is_available() else '')
print(f'Model: {MODEL_ID}')
print(f'QLoRA: {USE_QLORA}')

## 1. Load Dataset

In [ ]:
from datasets import Dataset

def load_jsonl(path):
    records = []
    with open(path) as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))
    return records

train_data = load_jsonl(PROC_DIR / 'sft_train.jsonl')
val_data   = load_jsonl(PROC_DIR / 'sft_val.jsonl')

print(f'Train: {len(train_data):,} examples')
print(f'Val:   {len(val_data):,} examples')
print('\nSample:')
sample = train_data[0]
for msg in sample['messages']:
    print(f"  [{msg['role']}]: {msg['content'][:120]}...")

## 2. Load Model + Tokenizer

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.padding_side = 'right'
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# QLoRA config: load model in 4-bit NF4 quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
) if USE_QLORA else None

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    torch_dtype=torch.bfloat16 if not USE_QLORA else None,
    trust_remote_code=True,
    use_cache=False,
)

if USE_QLORA:
    model = prepare_model_for_kbit_training(model)

print(f'Model loaded. Parameters: {sum(p.numel() for p in model.parameters())/1e9:.1f}B')

## 3. Configure LoRA

In [ ]:
# Target modules for Qwen2.5 architecture
# These are the attention projection matrices + MLP — where domain knowledge is stored
QWEN_LORA_TARGETS = [
    'q_proj', 'k_proj', 'v_proj', 'o_proj',  # attention
    'gate_proj', 'up_proj', 'down_proj',       # MLP (FFN)
]

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=QWEN_LORA_TARGETS,
    task_type=TaskType.CAUSAL_LM,
    bias='none',
    inference_mode=False,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Sanity: check trainable param count
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'Trainable: {trainable/1e6:.1f}M / {total/1e9:.1f}B ({100*trainable/total:.2f}%)')

## 4. Tokenize Dataset

In [ ]:
def format_and_tokenize(examples):
    texts = []
    for msgs in examples['messages']:
        # Use Qwen's chat template
        text = tokenizer.apply_chat_template(
            msgs,
            tokenize=False,
            add_generation_prompt=False,
        )
        texts.append(text)
    
    tokenized = tokenizer(
        texts,
        truncation=True,
        max_length=MAX_SEQ_LEN,
        padding='max_length',
        return_tensors='pt',
    )
    
    # Mask prompt tokens: loss only on assistant response
    labels = tokenized['input_ids'].clone()
    for i, text in enumerate(texts):
        # Find where assistant response starts
        assistant_token = tokenizer.encode('<|im_start|>assistant', add_special_tokens=False)
        ids = tokenized['input_ids'][i].tolist()
        last_assistant_start = -1
        for j in range(len(ids) - len(assistant_token)):
            if ids[j:j+len(assistant_token)] == assistant_token:
                last_assistant_start = j + len(assistant_token)
        if last_assistant_start > 0:
            labels[i, :last_assistant_start] = -100
    
    tokenized['labels'] = labels
    return tokenized

train_ds = Dataset.from_list(train_data)
val_ds   = Dataset.from_list(val_data)

train_ds = train_ds.map(format_and_tokenize, batched=True, batch_size=32, remove_columns=train_ds.column_names)
val_ds   = val_ds.map(format_and_tokenize,   batched=True, batch_size=32, remove_columns=val_ds.column_names)

print(f'Tokenized train: {len(train_ds)} | val: {len(val_ds)}')

## 5. Train

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling
from datetime import datetime

run_name = f'qwen_rf_lora_{datetime.now().strftime("%Y%m%d_%H%M")}_r{LORA_R}'

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / 'checkpoints'),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type='cosine',
    weight_decay=0.01,
    bf16=True,
    gradient_checkpointing=True,
    optim='paged_adamw_8bit' if USE_QLORA else 'adamw_torch',
    logging_steps=20,
    eval_strategy='steps',
    eval_steps=200,
    save_strategy='epoch',
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    report_to='wandb',
    run_name=run_name,
    ddp_find_unused_parameters=False,
    dataloader_num_workers=4,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
)

print(f'Starting fine-tuning run: {run_name}')
trainer.train()

## 6. Save LoRA Adapter

In [ ]:
adapter_dir = OUTPUT_DIR / 'adapter'
model.save_pretrained(str(adapter_dir))
tokenizer.save_pretrained(str(adapter_dir))
print(f'LoRA adapter saved to {adapter_dir}')

# List adapter files
for f in sorted(adapter_dir.rglob('*')):
    if f.is_file():
        print(f'  {f.name}  ({f.stat().st_size/1e6:.1f} MB)')

## 7. Quick Inference Test

In [ ]:
from peft import PeftModel

# Load merged model for inference
model.eval()

TEST_QUESTIONS = [
    'Design a 28 GHz LNA in 65nm CMOS targeting NF < 3 dB. What transistor width do you recommend and why?',
    'Write GLayout Python code to generate a 28 GHz microstrip transmission line on GF180MCU.',
    'What is the noise figure of a cascaded system with an LNA (G=20dB, NF=3dB) followed by a mixer (NF=10dB)?',
    'Design a Ka-band patch antenna for a CubeSat at 28 GHz. What substrate and dimensions would you use?',
]

for q in TEST_QUESTIONS:
    messages = [
        {'role': 'system', 'content': 'You are an expert RF IC and antenna engineer specializing in Ka-band CMOS chip design for satellite communications.'},
        {'role': 'user', 'content': q},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors='pt').to(model.device)
    
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    
    response = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    print(f'Q: {q[:80]}...')
    print(f'A: {response[:400]}')
    print('---')

## 8. (Optional) Merge LoRA into Base Model for Deployment

In [ ]:
# Merge adapter weights into base model (needed for vLLM deployment)
# WARNING: requires loading full model in fp16/bf16 — needs ~14 GB VRAM for 7B

MERGE = False  # Set True when ready to deploy

if MERGE:
    from peft import PeftModel
    from transformers import AutoModelForCausalLM
    
    base = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.bfloat16, device_map='auto')
    merged = PeftModel.from_pretrained(base, str(adapter_dir))
    merged = merged.merge_and_unload()
    
    merged_dir = OUTPUT_DIR / 'merged'
    merged.save_pretrained(str(merged_dir))
    tokenizer.save_pretrained(str(merged_dir))
    print(f'Merged model saved to {merged_dir}')